In [ ]:
# ==============================================================================
# LABORATÓRIO ECONOMÉTRICO: TESTES DE CAUSALIDADE E DINÂMICA COMPARATIVA (PT)
# VIX TUPINIQUIM III (TVP-VAR) vs. EPU BRASIL (BAKER, BLOOM & DAVIS)
# ==============================================================================

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

warnings.filterwarnings('ignore')

plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)

def formatar_molduras(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')
    ax.spines['right'].set_color('black')

print('=' * 80)
print(
    '--- [BANCADA ECONOMÉTRICA]: TESTES DE CAUSALIDADE (VIX TUPINIQUIM III vs EPU BRASIL) ---'
)
print('=' * 80)

# ==============================================================================
# 1. CARGA E ALINHAMENTO DAS SÉRIES HISTÓRICAS
# ==============================================================================
print('\n[1/5] Carregando bases de dados (EPU Brasil e VIX Tupiniquim III)...')

df_epu = pd.read_excel('epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index')
df_epu['Data'] = pd.to_datetime(
    df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01'
)
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

# Leitura da série histórica gerada pelo pipeline do VIX III
try:
    df_vix = pd.read_excel('vix_tupiniquim_iii_series_historicas.xlsx')
except Exception:
    df_vix = pd.read_csv('vix_tupiniquim_iii_series_historicas.csv', sep=';')

df_vix['Data'] = pd.to_datetime(df_vix['Data'])

# ==============================================================================
# 2. FILTRO ESTRUTURAL (STL) E MERGE
# ==============================================================================
print('\n[2/5] Aplicando Filtro Estrutural STL (period=13) no EPU Brasil...')
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

# Alinhamento da amostra comum
col_vix_alvo = 'VIX_Tupiniquim_III_Media100'

df_analise = (
    pd.merge(
        df_vix[['Data', col_vix_alvo]],
        df_epu[['Data', 'EPU_SA']],
        on='Data',
        how='inner',
    )
    .sort_values('Data')
    .reset_index(drop=True)
)
print(
    f'[ALINHAMENTO]: Amostra comum sincronizada com {len(df_analise)} observações mensais.'
)

# ==============================================================================
# 3. VERIFICAÇÃO DE ESTACIONARIEDADE (ADF) & DIFERENCIAÇÃO
# ==============================================================================
print('\n' + '=' * 80)
print('                    1. TESTES DE ESTACIONARIEDADE (ADF)')
print('=' * 80)

df_analise['dVIX_III'] = df_analise[col_vix_alvo].diff()
df_granger = df_analise.dropna().reset_index(drop=True)

p_vix_level = adfuller(df_analise[col_vix_alvo])[1]
p_vix_diff = adfuller(df_granger['dVIX_III'])[1]
p_epu_level = adfuller(df_analise['EPU_SA'])[1]

print(
    f"-> VIX Tupiniquim III (Nível)        | P-valor ADF: {p_vix_level:.4f} -> {'Estacionário I(0)' if p_vix_level < 0.05 else 'Não-Estacionário I(1)'}"
)
print(
    f"-> Delta VIX Tupiniquim III (Diff 1) | P-valor ADF: {p_vix_diff:.4f} -> {'Estacionário I(0)' if p_vix_diff < 0.05 else 'Não-Estacionário I(1)'}"
)
print(
    f"-> EPU Dessazonalizado (Nível)       | P-valor ADF: {p_epu_level:.4f} -> {'Estacionário I(0)' if p_epu_level < 0.05 else 'Não-Estacionário I(1)'}"
)
print('=' * 80)

# ==============================================================================
# 4. TESTE DE CAUSALIDADE DE GRANGER ESTACIONÁRIO (Delta VIX III vs EPU_SA)
# ==============================================================================
max_lags = 3
print('\n' + '=' * 80)
print(
    f'        2. TESTE DE GRANGER PADRÃO COM SÉRIES ESTACIONÁRIAS [Delta VIX vs EPU] (Lags 1 a {max_lags})'
)
print('=' * 80)

# Sentido 1: Delta VIX III -> EPU_SA
print('\n[SENTIDO 1]: Variação do VIX III (Mercado) -> EPU (Notícias/Imprensa)')
gc_vix_to_epu = grangercausalitytests(
    df_granger[['EPU_SA', 'dVIX_III']], maxlag=max_lags, verbose=False
)
for lag in range(1, max_lags + 1):
    p_val = gc_vix_to_epu[lag][0]['ssr_ftest'][1]
    status = 'REJEITA H0 (Granger-Causa)' if p_val < 0.05 else 'Não Rejeita H0 (Sem Causalidade)'
    print(f'-> Lag {lag}: P-valor = {p_val:.4f} | {status}')

# Sentido 2: EPU_SA -> Delta VIX III
print('\n[SENTIDO 2]: EPU (Notícias/Imprensa) -> Variação do VIX III (Mercado)')
gc_epu_to_vix = grangercausalitytests(
    df_granger[['dVIX_III', 'EPU_SA']], maxlag=max_lags, verbose=False
)
for lag in range(1, max_lags + 1):
    p_val = gc_epu_to_vix[lag][0]['ssr_ftest'][1]
    status = 'REJEITA H0 (Granger-Causa)' if p_val < 0.05 else 'Não Rejeita H0 (Sem Causalidade)'
    print(f'-> Lag {lag}: P-valor = {p_val:.4f} | {status}')
print('=' * 80)

# ==============================================================================
# 5. PROCEDIMENTO DE TODA-YAMAMOTO (1995)
# ==============================================================================
print('\n' + '=' * 80)
print('            3. TESTE DE CAUSALIDADE ROBUSTO DE TODA-YAMAMOTO')
print('=' * 80)

d_max = 1
var_model = VAR(df_analise[[col_vix_alvo, 'EPU_SA']])
lag_order = var_model.select_order(maxlags=6)
k = max(lag_order.bic, 1)

print(f'-> Ordem ótima do VAR em nível (k via BIC): {k} lag(s)')
print(f'-> Ordem máxima de integração assumida (d_max): {d_max}')
print(f'-> VAR aumentado estimado com (k + d_max) = {k + d_max} lags em nível.\n')

def executar_toda_yamamoto(df_data, y_nome, x_nome, k_lags, d_integracao):
    df_ty = pd.DataFrame(index=df_data.index)
    df_ty['const'] = 1.0

    for i in range(1, k_lags + d_integracao + 1):
        df_ty[f'{y_nome}_lag{i}'] = df_data[y_nome].shift(i)

    for i in range(1, k_lags + d_integracao + 1):
        df_ty[f'{x_nome}_lag{i}'] = df_data[x_nome].shift(i)

    df_ty['target'] = df_data[y_nome]
    df_reg = df_ty.dropna()

    X_mat = df_reg.drop(columns=['target'])
    y_vec = df_reg['target']

    modelo_ols = sm.OLS(y_vec, X_mat).fit(cov_type='HC1')

    restricoes = [f'{x_nome}_lag{i} = 0' for i in range(1, k_lags + 1)]
    formula_wald = ', '.join(restricoes)

    teste_wald = modelo_ols.wald_test(formula_wald, scalar=True)
    return teste_wald.statistic, teste_wald.pvalue

stat_1, p_val_1 = executar_toda_yamamoto(
    df_analise, 'EPU_SA', col_vix_alvo, k, d_max
)
status_1 = 'REJEITA H0 (Causa)' if p_val_1 < 0.05 else 'Não Rejeita H0 (Sem Causalidade)'
print('[SENTIDO 1 (TY)]: VIX Tupiniquim III -> EPU_SA (Nível)')
print(f'-> Estatística Wald: {stat_1:.4f} | P-valor = {p_val_1:.4f} | {status_1}')

stat_2, p_val_2 = executar_toda_yamamoto(
    df_analise, col_vix_alvo, 'EPU_SA', k, d_max
)
status_2 = 'REJEITA H0 (Causa)' if p_val_2 < 0.05 else 'Não Rejeita H0 (Sem Causalidade)'
print('\n[SENTIDO 2 (TY)]: EPU_SA (Nível) -> VIX Tupiniquim III')
print(f'-> Estatística Wald: {stat_2:.4f} | P-valor = {p_val_2:.4f} | {status_2}')
print('=' * 80)

# ==============================================================================
# 6. GRÁFICO COMPARATIVO: VIX TUPINIQUIM III vs. EPU BRASIL (DUPLO EIXO)
# ==============================================================================
print('\nGerando Gráfico Comparativo: VIX Tupiniquim III vs. EPU Brasil (Dois Eixos)...')

fig, ax1 = plt.subplots(figsize=(15, 5.5), dpi=300)

ax1.plot(
    df_analise['Data'],
    df_analise[col_vix_alvo],
    color='#1b5e20',
    linewidth=2.4,
    label='VIX Tupiniquim III (Base 100)',
)
ax1.set_xlabel('Ano', fontweight='bold', fontsize=10)
ax1.set_ylabel(
    'VIX Tupiniquim III (Base Média = 100)',
    fontweight='bold',
    color='#1b5e20',
    fontsize=11,
)
ax1.tick_params(axis='y', labelcolor='#1b5e20')
ax1.axhline(100, color='#1b5e20', linestyle=':', linewidth=1.0, alpha=0.6)
ax1.grid(True, linestyle=':', alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(
    df_analise['Data'],
    df_analise['EPU_SA'],
    color='#1f77b4',
    linewidth=1.9,
    linestyle='--',
    label='EPU Brasil (Dessazonalizado)',
)
ax2.set_ylabel(
    'EPU Index (Baker, Bloom & Davis)',
    fontweight='bold',
    color='#1f77b4',
    fontsize=11,
)
ax2.tick_params(axis='y', labelcolor='#1f77b4')

formatar_molduras(ax1)
formatar_molduras(ax2)

# Unificação de legendas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=False, fontsize=9.5)

plt.tight_layout()
plt.savefig('vix_iii_vs_epu_brasil_pt.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n[SUCESSO]: Pipeline executado e 'vix_iii_vs_epu_brasil_pt.png' salvo com sucesso!")